# 01 — Dataset Exploration & Data Preparation (MS COCO 2017)

**Project**: Black-and-White Image Colorization with Deep Learning  

This notebook verifies our reproducible 12,000-image MS COCO 2017 subset, grayscale filtering, 80/10/10 train/validation/test split, CIELAB normalization to `[-1, 1]`, and exploratory data analysis (EDA).

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.dataset import ColorizationDataset
from src.utils import lab_normalized_to_rgb

print('Project root:', ROOT)

## 1. Dataset Statistics & Split Integrity Check

In [ ]:
stats_path = ROOT / 'data' / 'splits' / 'dataset_stats.json'
with open(stats_path, 'r', encoding='utf-8') as f:
    stats = json.load(f)

print(json.dumps(stats, indent=2))

train_df = pd.read_csv(ROOT / 'data' / 'splits' / 'train.csv')
val_df = pd.read_csv(ROOT / 'data' / 'splits' / 'val.csv')
test_df = pd.read_csv(ROOT / 'data' / 'splits' / 'test.csv')

assert set(train_df['id']).isdisjoint(set(val_df['id'])), 'Overlap between train and val!'
assert set(train_df['id']).isdisjoint(set(test_df['id'])), 'Overlap between train and test!'
assert set(val_df['id']).isdisjoint(set(test_df['id'])), 'Overlap between val and test!'

split_summary = pd.DataFrame([
    {'Split': 'Train', 'Count': len(train_df), 'Percentage': f'{len(train_df)/12000*100:.1f}%', 'Augmentation': 'Random Horizontal Flip (p=0.5)'},
    {'Split': 'Validation', 'Count': len(val_df), 'Percentage': f'{len(val_df)/12000*100:.1f}%', 'Augmentation': 'None'},
    {'Split': 'Test', 'Count': len(test_df), 'Percentage': f'{len(test_df)/12000*100:.1f}%', 'Augmentation': 'None'},
])
display(split_summary)

## 2. CIELAB Preprocessing & Tensor Verification

In [ ]:
ds = ColorizationDataset(
    csv_path=ROOT / 'data' / 'splits' / 'train.csv',
    image_dir=ROOT / 'data' / 'processed_256',
    image_size=256,
    augment=True,
)

L, ab, fname = ds[0]
print(f'Sample file : {fname}')
print(f'L  tensor   : shape={tuple(L.shape)}, dtype={L.dtype}, min={L.min():.4f}, max={L.max():.4f}')
print(f'ab tensor   : shape={tuple(ab.shape)}, dtype={ab.dtype}, min={ab.min():.4f}, max={ab.max():.4f}')
assert L.shape == (1, 256, 256) and ab.shape == (2, 256, 256)
assert -1.01 <= L.min() and L.max() <= 1.01
assert -1.01 <= ab.min() and ab.max() <= 1.01
print('All tensor shape and normalization checks passed!')

## 3. Exploratory Data Analysis (EDA) Visuals

In [ ]:
for title, rel_path in [
    ('CIELAB & Resolution Distributions', 'results/plots/dataset_eda.png'),
    ('Sample Input (L) vs Target (RGB) Pairs', 'results/plots/dataset_samples.png'),
]:
    img_path = ROOT / rel_path
    if img_path.exists():
        img = Image.open(img_path)
        plt.figure(figsize=(14, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title)
        plt.show()
    else:
        print('Missing plot:', img_path)